# 1. NHẬP DỮ LIỆU VÀ KIỂM TRA TỔNG QUAN

In [ ]:
import pandas as pd
import numpy as np

path = "../data" 
df = pd.read_csv(f"{path}/tiki_cosmetics_raw.csv")

print(f"Shape: {df.shape}")
print(f"\n--- Kiểu dữ liệu ---")
print(df.dtypes)
print(f"\n--- Tỉ lệ missing (%) ---")
missing = (df.isnull().mean() * 100).round(1)
print(missing[missing > 0].sort_values(ascending=False))
print(f"\n--- Thống kê cơ bản ---")
print(df[["price","original_price","discount_rate","rating","review_count","sold_count"]].describe())

# 2. LẤP ĐẦY DỮ LIỆU THIẾU VÀ CHUẨN HÓA CƠ BẢN

In [ ]:
# 1. origin: điền từ is_imported nếu trống
def fill_origin(row):
    if pd.notna(row["origin"]):
        return row["origin"]
    if row["is_imported"] == True:
        return "Ngoài nước (không rõ)"
    if row["is_imported"] == False:
        return "Việt Nam"
    return np.nan

df["origin"] = df.apply(fill_origin, axis=1)

# 2. rating & review_count: sản phẩm mới chưa có đánh giá → điền 0
df["rating"]       = df["rating"].fillna(0)
df["review_count"] = df["review_count"].fillna(0).astype(int)
df["sold_count"]   = df["sold_count"].fillna(0).astype(int)

# 3. brand_name: điền "Không rõ"
df["brand_name"] = df["brand_name"].fillna("Không rõ")

# 4. discount_rate: nếu thiếu thì tính lại từ giá
df["discount_rate"] = df.apply(
    lambda r: r["discount_rate"] if pd.notna(r["discount_rate"]) and r["discount_rate"] > 0
              else (round((r["original_price"] - r["price"]) / r["original_price"] * 100, 1)
                   if r["original_price"] > r["price"] else 0),
    axis=1
)

print("Missing sau xử lý:")
print((df.isnull().mean() * 100).round(1)[lambda x: x > 0])

# 3. NHẬN DIỆN VÀ LOẠI BỎ DỮ LIỆU BẤT THƯỜNG

In [ ]:
# Kiểm tra outlier trước
print("--- Giá ---")
print(df["price"].describe())
print(f"\nSản phẩm giá > 50 triệu: {(df['price'] > 50_000_000).sum()}")
print(f"Sản phẩm giá = 0: {(df['price'] == 0).sum()}")

print("\n--- Discount rate ---")
print(df["discount_rate"].describe())
print(f"Discount > 90%: {(df['discount_rate'] > 90).sum()}")

# Xử lý: loại bỏ sản phẩm bất thường
df = df[df["price"] > 0]                    # giá = 0 là lỗi
df = df[df["price"] <= 50_000_000]          # nước hoa xa xỉ hợp lý
df = df[df["discount_rate"].between(0, 90)] # discount > 90% thường là lỗi

print(f"\nSau xử lý outlier: {len(df):,} sản phẩm")

# 4. CHUẨN HÓA XUẤT XỨ VÀ PHÂN LOẠI THỊ TRƯỜNG

In [ ]:
# Chuẩn hóa các tên gọi khác nhau về tên chuẩn
ORIGIN_MAP = {
    # Việt Nam
    "việt nam": "Việt Nam", "viet nam": "Việt Nam",
    "china / vietnam": "Việt Nam", "việt nam / đài loan": "Việt Nam",
    "nhật bản / việt nam": "Việt Nam", "anh, việt nam": "Việt Nam",

    # Hàn Quốc
    "hàn quốc": "Hàn Quốc", "korea": "Hàn Quốc",
    "hàn quốc / trung quốc": "Hàn Quốc",

    # Nhật Bản
    "nhật bản": "Nhật Bản", "japan": "Nhật Bản",

    # Trung Quốc
    "trung quốc": "Trung Quốc", "china": "Trung Quốc",
    "hk/tq": "Trung Quốc",

    # Các nước khác
    "mỹ": "Mỹ", "usa": "Mỹ",
    "pháp": "Pháp", "france": "Pháp",
    "thái lan": "Thái Lan", "thailand": "Thái Lan",
    "đức": "Đức", "germany": "Đức",
    "anh": "Anh", "uk": "Anh",
    "đài loan": "Đài Loan", "taiwan": "Đài Loan",
    "hong kong": "Hồng Kông", "hk": "Hồng Kông",
    "indonesia": "Indonesia",
    "ấn độ": "Ấn Độ", "india": "Ấn Độ",
    "úc": "Úc", "australia": "Úc",
    "italia": "Ý", "italy": "Ý",
    "tây ban nha": "Tây Ban Nha",
    "p.r.c": "Trung Quốc", "prc": "Trung Quốc",
    "thụy sỹ": "Thụy Sỹ", "switzerland": "Thụy Sỹ",
    "đài loan": "Đài Loan", "taiwan": "Đài Loan",
    "belgium": "Bỉ", "bỉ": "Bỉ",
    "canada": "Canada",
}

VIETNAM_SET = {"Việt Nam"}

def normalize_origin(raw):
    if pd.isna(raw):
        return "Không rõ"
    raw_lower = str(raw).lower().strip()
    for key, val in ORIGIN_MAP.items():
        if key in raw_lower:
            return val
    return str(raw).strip()  # giữ nguyên nếu không match

def classify_origin(origin_norm):
    if origin_norm in VIETNAM_SET:
        return "Trong nước"
    if origin_norm == "Không rõ":
        return "Không rõ"
    return "Ngoài nước"

df["origin_normalized"] = df["origin"].apply(normalize_origin)
df["origin_class"]      = df["origin_normalized"].apply(classify_origin)

# Kiểm tra
print("=== Xuất xứ sau chuẩn hóa ===")
print(df["origin_normalized"].value_counts().head(20))
print(f"\n=== Phân loại trong/ngoài nước ===")
print(df["origin_class"].value_counts())
print(f"\nTỉ lệ: {df['origin_class'].value_counts(normalize=True).mul(100).round(1).to_string()}")

# 5. XÂY DỰNG CÁC CHỈ SỐ PHÂN TÍCH CHIỀU SÂU

In [ ]:
# 1. Phân khúc giá
def price_segment(price):
    if price < 100_000:    return "Dưới 100k"
    if price < 300_000:    return "100k – 300k"
    if price < 700_000:    return "300k – 700k"
    if price < 2_000_000:  return "700k – 2tr"
    return "Trên 2tr"

df["price_segment"] = df["price"].apply(price_segment)

# 2. Mức độ phổ biến dựa trên lượt bán
def popularity_tier(sold):
    if sold == 0:      return "Chưa có lượt bán"
    if sold < 50:      return "Mới"
    if sold < 500:     return "Phổ biến"
    if sold < 2000:    return "Bán chạy"
    return "Bestseller"

df["popularity_tier"] = df["sold_count"].apply(popularity_tier)

# 3. Mức độ đánh giá
def rating_tier(r):
    if r == 0:    return "Chưa có đánh giá"
    if r < 3.5:   return "Thấp (< 3.5)"
    if r < 4.5:   return "Trung bình (3.5–4.5)"
    return "Cao (≥ 4.5)"

df["rating_tier"] = df["rating"].apply(rating_tier)

# 4. Có giảm giá hay không
df["has_discount"] = df["discount_rate"] > 0

# 5. Tạo cột Product_Type (Phân loại nhóm lớn)
type_map = {
    # 1. Skincare (Chăm sóc da mặt)
    "Sữa rửa mặt": "Skincare", "Tẩy trang": "Skincare", "Toner": "Skincare", 
    "Serum": "Skincare", "Kem dưỡng da": "Skincare", "Mặt nạ": "Skincare",
    "Kem chống nắng (mặt)": "Skincare", "Trị mụn & sẹo": "Skincare", 
    "Chống lão hóa": "Skincare", "Tẩy tế bào chết mặt": "Skincare", "Xịt khoáng": "Skincare",
    "Dưỡng mắt": "Skincare", "Bông tẩy trang": "Skincare", "Giấy thấm dầu": "Skincare",
    "Máy rửa mặt": "Skincare", "Máy hút mụn": "Skincare",

    # 2. Makeup & Nail (Trang điểm & Móng)
    "Son môi": "Makeup", "Trang điểm mặt": "Makeup", "Trang điểm mắt": "Makeup",
    "Dụng cụ trang điểm": "Makeup", "Bộ trang điểm": "Makeup", "Trang điểm khác": "Makeup",
    "Chăm sóc móng": "Makeup", "Dụng cụ làm đẹp khác": "Makeup",

    # 3. Body & Personal Care (Chăm sóc cơ thể & Vệ sinh)
    "Sữa tắm": "Body Care", "Dưỡng thể": "Body Care", "Kem chống nắng (thể)": "Body Care",
    "Tẩy tế bào chết body": "Body Care", "Bộ chăm sóc toàn thân": "Body Care",
    "Sản phẩm sạch khuẩn": "Body Care", "Massage toàn thân": "Body Care",
    "Sản phẩm tẩy lông": "Body Care", "Lăn, xịt khử mùi": "Body Care",
    "Dung dịch vệ sinh": "Body Care", "Răng miệng": "Body Care",
    "Bàn chải": "Body Care", "Kem đánh răng": "Body Care", "Tẩy trắng răng": "Body Care",

    # 4. Hair Care (Chăm sóc tóc)
    "Dầu gội & xả": "Hair Care", "Dưỡng tóc": "Hair Care", "Tạo kiểu tóc": "Hair Care",
    "Thuốc nhuộm tóc": "Hair Care", "Bộ chăm sóc tóc": "Hair Care",

    # 5. Fragrance (Nước hoa)
    "Nước hoa nữ": "Fragrance", "Nước hoa nam": "Fragrance", 
    "Xịt thơm cơ thể (Body Mist)": "Fragrance",

    # 6. Supplements (Thực phẩm chức năng làm đẹp)
    "TPCN làm đẹp": "Supplements", "Vitamin": "Supplements", "Thải độc": "Supplements"
}

df["product_type"] = df["category"].map(type_map).fillna("Khác")

# 6. Tính tỉ lệ Review_Ratio
# Công thức: Review_Ratio = Review_Count/Sold_Count
# Nếu Sold_Count = 0 thì tỉ lệ = 0 để tránh lỗi chia cho 0
df["review_ratio"] = df.apply(
    lambda r: round(r["review_count"] / r["sold_count"], 4) if r["sold_count"] > 0 else 0, 
    axis=1
)

# 7. Tính doanh thu ước tính: Doanh thu = Giá hiện tại * Lượt bán
df["estimated_revenue"] = df["price"] * df["sold_count"]

print("=== Phân khúc giá ===")
print(df["price_segment"].value_counts())
print("\n=== Mức độ phổ biến ===")
print(df["popularity_tier"].value_counts())
print("\n=== Đánh giá ===")
print(df["rating_tier"].value_counts())
print("=== Phân loại nhóm hàng (Product Type) ===")
print(df["product_type"].value_counts())
print("\n=== Tỉ lệ Review trung bình theo nhóm ===")
print(df.groupby("product_type")["review_ratio"].mean())

# 6. TỔNG KẾT VÀ XUẤT DỮ LIỆU SẠCH

In [ ]:
df_clean = df.copy()

# Sắp xếp lại cột
cols_order = [
    # 1. Nhóm định danh (Identification) - Luôn để đầu tiên
    "product_id", "name", "brand_name", "seller_name",
    
    # 2. Nhóm Phân loại & Xuất xứ (Core của đề tài)
    "product_type", "category", "primary_category", 
    "origin_class", "origin_normalized", "origin", "is_imported",
    
    # 3. Nhóm Tài chính & Giá cả
    "price", "original_price", "discount_rate", "has_discount", "price_segment",
    
    # 4. Nhóm Hiệu quả kinh doanh & Thị hiếu (Metrics)
    "sold_count", "popularity_tier", "review_count", "review_ratio", "rating", "rating_tier", "estimated_revenue",
    
    # 5. Nhóm Chỉ số niềm tin & Trạng thái (Trust & Status)
    "is_official_store", "is_authentic", "has_authentic_badge", "tiki_verified", "availability"
]
df_clean = df_clean[[c for c in cols_order if c in df_clean.columns]]
path = "../data"
df_clean.to_csv(f"{path}/tiki_cosmetics_processed.csv", index=False, encoding="utf-8-sig")

print(f"=== ĐÃ LƯU tiki_cosmetics.csv ===")
print(f"Shape: {df_clean.shape}")
print(f"\n--- Preview ---")
print(df_clean[["name","origin_class","price_segment","rating","sold_count"]].head(8).to_string())